# Local action consistency

LAC@8 of a key (offline, held-out demos) vs the closed-loop success of a lookup with it: one point per backbone key, before and after fine-tuning.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

import utils as u

FS_SMALL = 9  # pt: \small of the paper
PAGE_WIDTH = 5.5  # in: \textwidth of the paper (\linewidth of a figure)
ROOT = Path.cwd().parent
data_path = ROOT / "results"
figs_path = ROOT / "analysis/figs"
style_path = ROOT / "analysis/figure.mplstyle"

In [ ]:
data = json.loads((data_path / "data.json").read_text())
figs_path.mkdir(exist_ok=True)
plt.style.use(style_path)

In [ ]:
NEIGHBOR_COLOUR = {"generic": u.Colour.HUMAN_DARK, "learnt": u.Colour.KNN, "before": u.Colour.BEFORE, "after": u.Colour.AFTER}

In [ ]:
pts = data["neighbor_metric"]["correlation"]["points"]
roles = dict(u.NEIGHBOR_KEYS)
for k in data["extra_keys"]:  # the chosen layer of every other backbone
    roles[k["label"]] = "after" if " LIBERO " in k["label"] else "before"
fair = [(v["lac@8"], v["success"]) for n, v in pts.items() if v["fair"] and n in roles and v["lac@8"] >= 0.7]
xs, ys = np.array(fair).T
slope, icpt = np.polyfit(xs, ys, 1)
gx = np.array([min(xs) - 0.01, max(xs) + 0.01])

In [ ]:
fig, ax = plt.subplots(figsize=(PAGE_WIDTH, 1.872), layout="constrained")
ax.plot(gx, slope * gx + icpt, color=u.Colour.AXIS, linestyle=(0, (4, 3)), linewidth=1.4, zorder=1)
off = []  # keys far off the scale of the others (a broken key), noted instead of plotted
for name, role in roles.items():
    if name in pts:
        v = pts[name]
        if v["lac@8"] < 0.7:
            off.append((name, v))
            continue
        ax.scatter([v["lac@8"]], [v["success"]], s=95, color=NEIGHBOR_COLOUR[role], edgecolor="white", linewidth=1.2,
                   zorder=3)  # fmt: skip
for i, (name, v) in enumerate(off):
    label = r"$\pi_0$ fine-tuned" if name.startswith("π0 LIBERO") else name
    ax.annotate(f"← {label}: LAC {v['lac@8']:.2f}, {v['success']:.0f}% (off scale)", xy=(0.45, 0.06 + 0.1 * i),
                xycoords="axes fraction", fontsize=FS_SMALL)  # fmt: skip
for name, label, offset in (("ResNet-18 grid", "ImageNet ResNet-18", (-8, 5)),
                            ("ResNet-18 grid (learnt key)", "learnt key", (-8, 6)), ("DINOv2-S", "DINOv2", (8, -4))):  # fmt: skip
    ax.annotate(label, (pts[name]["lac@8"], pts[name]["success"]), xytext=offset, textcoords="offset points",
                ha="right" if offset[0] < 0 else "left", fontsize=FS_SMALL)  # fmt: skip
ax.set_xlabel("local action consistency LAC@8 (offline)")
ax.set_ylabel("lookup on LIBERO (%)")
ax.grid(True)
ax.set_axisbelow(True)
handles = [Line2D([], [], marker="o", linestyle="", color=NEIGHBOR_COLOUR[r], markersize=9, label=lab)
           for r, lab in (("generic", "ImageNet / DINOv2"), ("before", "VLM before fine-tuning"),
                          ("after", "after"), ("learnt", "learnt key"))]  # fmt: skip
fig.legend(handles=handles, frameon=False, loc="outside upper center", ncol=4,
           handletextpad=0.3, columnspacing=1.0)  # fmt: skip
fig.savefig(figs_path / "neighbors.pdf")